# 03-Structured Output & Pydantic Coercion

In Lesson 02, we bridged the Air Gap between the generative neural network and the deterministic Python server using Function Calling. We taught the LLM to format its intent as a JSON string so the server could execute a tool.

However, Function Calling is an *action-oriented* architecture. What if our goal isn't to trigger an external API, but rather to **extract complex, deeply nested data** from unstructured text? What if we need the Agent to synthesize a strict mathematical state object to pass to the next node in our LangGraph?

If we rely on prompt engineering alone (*"Please output valid JSON..."*), the LLM will eventually hallucinate a missing comma, append conversational filler (*"Here is your JSON:"*), or return a string instead of a float. This instantly destroys downstream pipelines.

To mathematically guarantee the structural integrity of our Agent's thoughts, we must enforce **Structured Outputs via Pydantic Coercion**.

Let's set up our environment to engineer strict memory schemas.

In [ ]:
import json
from pydantic import BaseModel, Field, ValidationError
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
import numpy as np

# Set professional visualization styling
sns.set_theme(style="whitegrid")

print("✅ Systems Architecture & Pydantic Coercion Environment Ready.")

# 1. The Mathematics of Type Coercion

An LLM's autoregressive generation operates over an infinite space of text strings, which we define as $\Sigma^*$.
When building an Enterprise application, we do not want infinite possibilities. We want our output to belong to a strictly bounded, finite subset of valid JSON objects, which we define as $\mathcal{V}$.

If we rely purely on the LLM's raw policy $\pi_\theta$, the probability of generating a perfectly valid object $v \in \mathcal{V}$ decays exponentially as the complexity of the schema increases.

To solve this, we introduce **Pydantic** as a mathematical mapping function $f_{coerce}$:


$$f_{coerce}: \Sigma^* \rightarrow \mathcal{V} \cup \{\text{Error}\}$$

Pydantic acts as an impenetrable topological boundary. It reads the raw strings generated by the LLM and attempts to structurally coerce them into the expected tensors and types.

* If the LLM generates `"age": "25"`, Pydantic intercepts the string and forces it into an integer `25`.
* If the LLM generates `"confidence": 1.5`, Pydantic mathematically rejects it (if bounded between $0.0$ and $1.0$) and throws a `ValidationError`, which we can feed *back* into the LLM, creating a self-healing retry loop.

# 2. Architecting the Extraction Schema

Let's build an Enterprise data extraction pipeline. We will define a strict Pydantic schema for parsing financial contracts, simulate the messy, unstructured output of an LLM, and execute the coercion physics.

In [ ]:
# --- 🧱 The Pydantic Coercion Engine ---

# 1. Define the Strict Mathematical Boundary
class FinancialContractState(BaseModel):
    company_name: str = Field(..., description="The exact legal entity name.")
    transaction_value_usd: float = Field(..., gt=0, description="Total value in USD. Must be strictly positive.")
    is_recurring: bool = Field(..., description="True if this is a subscription or multi-year contract.")
    risk_score: int = Field(..., ge=1, le=10, description="Calculated risk from 1 to 10.")

def parse_llm_output(raw_llm_string: str) -> FinancialContractState:
    """Attempts to coerce the LLM's raw text into a strict mathematical object."""
    try:
        # Step 1: Strip conversational filler (The "Here is your JSON" problem)
        start_idx = raw_llm_string.find('{')
        end_idx = raw_llm_string.rfind('}') + 1
        clean_json_string = raw_llm_string[start_idx:end_idx]
        
        # Step 2: Parse raw string to Python dictionary
        raw_dict = json.loads(clean_json_string)
        
        # Step 3: The Pydantic Coercion Event
        validated_state = FinancialContractState(**raw_dict)
        return validated_state
        
    except json.JSONDecodeError:
        raise ValueError("FATAL: LLM failed to generate valid JSON syntax.")
    except ValidationError as e:
        raise ValueError(f"FATAL: Pydantic rejected the data topology.\n{e}")

# 2. Simulate LLM Generation Cases

# Case A: The "Messy but Coerce-able" LLM Output
# Notice the LLM added conversational text, passed the float as a string, and used 'true' (lowercase).
llm_output_a = """
Absolutely! Based on the contract, here is the structured data you requested:
{
    "company_name": "Nexus Dynamics LLC",
    "transaction_value_usd": "150000.50",
    "is_recurring": true,
    "risk_score": 4
}
Hope this helps!
"""

# Case B: The "Mathematically Invalid" LLM Output
# The LLM hallucinated a negative value and a risk score of 15.
llm_output_b = """
{
    "company_name": "CyberDyne Systems",
    "transaction_value_usd": -5000,
    "is_recurring": false,
    "risk_score": 15
}
"""

print("--- 🛡️ Executing Coercion Pipeline ---")

print("\n[Executing Case A: The Messy Output]")
try:
    state_a = parse_llm_output(llm_output_a)
    print(f"✅ SUCCESS: Pydantic successfully coerced the data!")
    print(f"Data Type of transaction_value: {type(state_a.transaction_value_usd)} (Notice it is no longer a string!)")
    print(f"Cleaned Object: {state_a.model_dump()}")
except ValueError as e:
    print(e)

print("\n[Executing Case B: The Invalid Output]")
try:
    state_b = parse_llm_output(llm_output_b)
except ValueError as e:
    print(f"❌ INTERCEPTED: Pydantic protected the system from invalid mathematics:")
    print(e)

print("\n--- 💡 Engineering Insight ---")
print("Look at Case A. The LLM generated a string `'150000.50'` and added garbage text before and after the JSON. The parsing logic stripped the garbage, and Pydantic autonomously coerced the string into a valid Python float. The downstream agentic state is perfectly preserved.")
print("In Case B, Pydantic acted as a firewall. It detected that the risk score exceeded our absolute threshold of 10. In a real LangGraph setup, we would catch this ValidationError, inject the error message back into the prompt, and tell the LLM: 'Your previous output failed validation. Risk score must be <= 10. Try again.'")

# 3. Visualizing Probability Space Collapse

To understand why this is physically necessary for multi-agent workflows, we must visualize the collapse of the LLM's probability space.

An unconstrained LLM generates points across a vast, chaotic multidimensional space. Pydantic acts as a **Geometric Manifold**, forcing the outputs to either snap into a rigid, structured grid, or be entirely rejected before they can contaminate the system.

In [ ]:
# Visualizing the Pydantic Topology
fig, ax = plt.subplots(figsize=(12, 7))
fig.suptitle("Probability Space Collapse: Unstructured Output vs. Pydantic Coercion", fontsize=16, fontweight='bold')

ax.axis('off')

# 1. The Unstructured LLM Space (Chaos)
np.random.seed(42)
chaos_x = np.random.normal(3, 1.5, 100)
chaos_y = np.random.normal(6, 1.5, 100)
ax.scatter(chaos_x, chaos_y, color='#e74c3c', alpha=0.3, s=30, label="Unstructured LLM Tokens ($\\Sigma^*$)")

# Draw the chaos boundary
chaos_circle = patches.Ellipse((3, 6), 7, 5, angle=0, fill=False, color='#e74c3c', linestyle='--', linewidth=2)
ax.add_patch(chaos_circle)
ax.text(3, 8.8, "Raw Probabilistic Space\n(Conversational Output, Hallucinations, Invalid Types)", 
        ha='center', color='darkred', fontweight='bold', fontsize=10)

# 2. The Pydantic Structured Manifold (Order)
grid_x, grid_y = np.meshgrid(np.linspace(8, 12, 4), np.linspace(3, 7, 4))
ax.scatter(grid_x, grid_y, color='#2ecc71', s=100, marker='s', edgecolors='black', zorder=3, label="Valid Pydantic States ($\\mathcal{V}$)")

# Draw the strict boundary
pydantic_box = patches.Rectangle((7.5, 2.5), 5, 5, fill=True, color='#2ecc71', alpha=0.1, linewidth=3, edgecolor='#27ae60')
ax.add_patch(pydantic_box)
ax.text(10, 8, "The Strict Pydantic Manifold\n(Enforced Types, Bounded Geometry)", 
        ha='center', color='darkgreen', fontweight='bold', fontsize=10)

# 3. The Coercion Arrows (The Physics of f_coerce)
# Successful Coercion (Snapping to the grid)
ax.annotate("", xy=(8, 5.6), xytext=(4, 6), arrowprops=dict(arrowstyle="->", color="black", lw=2, connectionstyle="arc3,rad=-0.1"))
ax.text(6, 6.1, "Successful Coercion\n(String $\\rightarrow$ Float)", ha='center', fontsize=9, fontweight='bold', rotation=0)

# Intercepted Validation Error (Rejection)
ax.annotate("", xy=(7.5, 3.5), xytext=(4, 4), arrowprops=dict(arrowstyle="-|>", color="#c0392b", lw=3, connectionstyle="arc3,rad=0.2"))
ax.scatter(7.5, 3.5, color='#c0392b', s=200, marker='X', zorder=5)
ax.text(5.5, 3.2, "ValidationError\n(Risk Score > 10)", ha='center', fontsize=9, fontweight='bold', color='#c0392b')

plt.xlim(0, 13)
plt.ylim(1, 10)
ax.legend(loc="lower left", fontsize=11)

plt.tight_layout()
plt.show()

## Real-World Use Case or Analogy:

Think of the difference between standard LLM prompts and Pydantic Coercion like **Submitting a Customs Declaration at the Airport**:

* **Unstructured Output (The Blank Piece of Paper)**: You hand the traveler a blank piece of paper and say, *"Please write down everything you are bringing into the country, the exact monetary value of the goods, and whether you are carrying agricultural products."* * The traveler writes a three-page essay about their vacation. They write *"I bought a watch for a few hundred bucks."* They completely forget to mention agricultural products. The customs officer (the downstream pipeline) cannot process this. The system halts.
* **Pydantic Coercion (The Strict Customs Form)**: You hand the traveler a rigid, structured government form with three exact boxes.
* Box 1: [Item Name - Text]
* Box 2: [Value - USD Numeric Only]
* Box 3: [Carrying Agriculture? - Check YES or NO].
* If the traveler tries to write "a few hundred bucks" in Box 2, the digital system physically blocks them and highlights the box in red (ValidationError). They are forced to correct it to "300.00" before they can step through the gate. The customs officer receives perfectly formatted, mathematically sound data every single time.